# 🌫️ STC-HGAT — PM2.5 Forecasting Thailand
## Spatio-Temporal Contrastive Heterogeneous Graph Attention Network

**Based on**: Yang & Peng (2024) *Mathematics* 12(8), 1193  
**Adapted for**: Daily PM2.5 forecasting across 79 Thai monitoring stations

```
Input (B, N=79, T=7, F=50)
        │
   Feature Embedding  →  (B, N, T, H=128)
        │
   ┌────┴──────────────────────────────────┐
   │                                       │
[Module I] HyperGAT              [Module II] HGAT
Spatial Heterogeneous            Temporal Heterogeneous
Hypergraph                       Graph
 ├─ Proximity hyperedges          ├─ Sequential day-edges
 ├─ Region nodes (5 regions)      └─ Seasonal (weekly) edges
 └─ Semantic (corr) edges
   │                                       │
   h_s (B,N,H)                    h_t (B,N,H)
   └────────────┬──────────────────────────┘
                │  Sumpooling Fusion h = h_s + h_t
                │
   [Module III] Position Encoding + Soft Attention → session_repr
                │
   ┌────────────┴──────────────────────┐
   │                                   │
[Module IV] InfoNCE           [Module V] MLP → PM2.5
Contrastive Loss L_c           AW Loss L_r
   └────────────────────────────────────┘
                │
          L = L_r + λ · L_c
```

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.data.graph_builder import GraphBuilder
from src.data.dataset import load_and_prepare
from src.models.stc_hgat_model import STCHGAT, STCHGATModel
from src.training.evaluator import evaluate_all

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

## 1 · Load & Prepare Data (date-based split — no leakage)

In [ ]:
DATA_DIR = Path('../bkk-pm25-data-ingestion/data/gold/model_ready')

data = load_and_prepare(
    data_dir=DATA_DIR,
    start_date='2023-01-01',
    lookback=7,
    min_stations=20,
    train_ratio=0.70,
    val_ratio=0.15,
    verbose=True,
)

train_ds       = data['train_ds']
val_ds         = data['val_ds']
test_ds        = data['test_ds']
target_scaler  = data['target_scaler']
station_meta   = data['station_meta']
station_order  = data['station_order']
feature_cols   = data['feature_cols']

print(f'\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Node features: {train_ds.n_features}')

## 2 · Build Station Graph (Spatial + Semantic + Wind + Hyperedges)

In [ ]:
# Load PM2.5 history for semantic edges
df_all = pd.concat([
    pd.read_parquet(DATA_DIR / 'train.parquet'),
    pd.read_parquet(DATA_DIR / 'val.parquet'),
    pd.read_parquet(DATA_DIR / 'test.parquet'),
])
df_all['date'] = pd.to_datetime(df_all['date'])
df_all = df_all[df_all['date'] >= '2023-01-01']

# Build PM2.5 history matrix (N_stations, T_hist)
pm25_pivot = (
    df_all.groupby(['stationID', 'date'])['pm2_5_mean']
    .mean().unstack(fill_value=0)
    .reindex(station_order).fillna(0)
)
pm25_history = pm25_pivot.values   # (N, T_hist)

lats = station_meta.set_index('stationID').reindex(station_order)['lat'].values
lons = station_meta.set_index('stationID').reindex(station_order)['lon'].values

# Mean wind for initial graph (will be dynamic per day in production)
wind_u_mean = df_all.groupby('stationID')['wind_u10_mean'].mean().reindex(station_order).fillna(0).values
wind_v_mean = df_all.groupby('stationID')['wind_v10_mean'].mean().reindex(station_order).fillna(0).values

# Build graph
gb = GraphBuilder(
    lats=lats, lons=lons,
    station_ids=station_order,
    pm25_history=pm25_history,
    spatial_thresholds_km=(50.0, 100.0, 200.0),
    spatial_edge_km=150.0,
    corr_threshold=0.70,
)

graphs = gb.build(wind_u=wind_u_mean, wind_v=wind_v_mean)
print(gb.summary(graphs))

membership = graphs['membership']
n_regions  = graphs['n_regions']

# Incidence matrix: (N + n_regions, E_h)
H_base = graphs['hyperedges_incidence']   # (N, E_h)
H_pad  = torch.zeros(n_regions, H_base.shape[1])
H_inc  = torch.cat([H_base, H_pad], dim=0)   # (N + n_regions, E_h)

print(f'\nIncidence matrix H: {H_inc.shape}')
print(f'n_regions: {n_regions}')

## 3 · Instantiate STC-HGAT Model

In [ ]:
config = {
    'in_channels':        train_ds.n_features,
    'hidden':             128,
    'n_regions':          n_regions,
    'hypergat_layers':    2,
    'seq_len':            train_ds.seq_len,
    'dropout':            0.1,
    'contrastive_lambda': 0.1,
    'aw_gamma':           2.0,
    'extreme_threshold':  2.0,     # in normalised units (≈ PM2.5 > 100 µg/m³)
    # Training
    'epochs':             100,
    'batch_size':         8,
    'lr':                 1e-3,
    'weight_decay':       1e-4,
    'patience':           15,
    'grad_clip':          1.0,
    'device':             DEVICE,
}

model = STCHGATModel(config=config, H_inc=H_inc, membership=membership)
print(f'Parameters: {model.net.count_parameters():,}')

## 4 · Train

In [ ]:
import mlflow
mlflow.set_experiment('pm25_stc_hgat')

with mlflow.start_run(run_name='stc_hgat_v1'):
    mlflow.log_params({k: v for k, v in config.items() if not isinstance(v, torch.Tensor)})

    model.fit(train_ds, val_ds)

    # Evaluate
    y_pred_norm, y_true_norm = model.predict(test_ds)

    # Inverse transform to original PM2.5 scale
    y_pred = target_scaler.inverse_transform(y_pred_norm)
    y_true = target_scaler.inverse_transform(y_true_norm)

    metrics = evaluate_all(y_true, y_pred)
    mlflow.log_metrics(metrics)

print('\n── Test Metrics (original PM2.5 scale) ──')
for k, v in metrics.items():
    print(f'  {k:6s}: {v:.4f}')

if metrics['R2'] > 0.9:
    print('\n🎉 TARGET R² > 0.9 ACHIEVED!')
elif metrics['R2'] > 0.8:
    print(f'\n✅ Good result: R² = {metrics["R2"]:.4f}')

## 5 · Training Curves

In [ ]:
hist = model._history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist['train_loss'], label='Train', lw=2)
axes[0].plot(hist['val_loss'],   label='Val',   lw=2, ls='--')
axes[0].set(title='Combined Loss (L_r + λ·L_c)', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist['train_r2'], label='Train R²', lw=2)
axes[1].plot(hist['val_r2'],   label='Val R²',   lw=2, ls='--')
axes[1].axhline(0.9, color='red', ls=':', label='Target R²=0.9')
axes[1].set(title='R² Score', xlabel='Epoch', ylabel='R²', ylim=(0, 1))
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6 · Spatial Error Analysis

In [ ]:
import plotly.graph_objects as go

# Per-station RMSE — need to run predict with station tracking
# (simplified: use overall predictions)

fig = go.Figure(go.Scattergeo(
    lon=lons, lat=lats,
    mode='markers',
    marker=dict(size=10, color='steelblue'),
    text=station_order,
    hovertemplate='%{text}<extra></extra>',
))

# Add graph edges (spatial)
for s, d in zip(graphs['spatial_edges'][0].tolist(), graphs['spatial_edges'][1].tolist()):
    fig.add_trace(go.Scattergeo(
        lon=[lons[s], lons[d]], lat=[lats[s], lats[d]],
        mode='lines', line=dict(width=0.5, color='rgba(100,100,255,0.3)'),
        showlegend=False,
    ))

fig.update_layout(
    title='STC-HGAT Station Graph — Thailand',
    geo=dict(scope='asia', center=dict(lat=15, lon=101), projection_scale=5),
    height=600,
)
fig.show()

## 7 · Save Checkpoint

In [ ]:
results_dir = Path('../results/stc_hgat')
results_dir.mkdir(parents=True, exist_ok=True)

model.save(str(results_dir / 'stc_hgat_best.pt'))

with open(results_dir / 'metrics.json', 'w') as f:
    json.dump({
        'model': 'STC-HGAT',
        'paper': 'Yang & Peng (2024) Mathematics 12(8) 1193',
        'metrics': metrics,
        'config': {k: v for k, v in config.items() if not isinstance(v, torch.Tensor)},
    }, f, indent=2)

print('✅ All artifacts saved.')